### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 3
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from BioSWOT experiment
# 
**DESCRIPTION:**
This script performs Step 3: Application of the Lag Correction. 
It reads the linear regression models and regime classifications generated 
in Step 2. Based on the profile's specific regime (e.g., Leg 1 Normal, 
Leg 1 Anomaly, or Leg 2) and its average descent rate (w), it calculates 
the precise scan lag required. It then shifts the Temperature signal by 
interpolating it over the calculated lag and recalculates Practical Salinity.
It also includes an optional diagnostic plotting feature for specific profiles.
#
INPUT: QC NetCDF files (*_step1_qc.nc), lag models CSV, and regimes CSV.
#
OUTPUT: NetCDF files with lag-corrected Temperature and Salinity (*_step3_lag.nc).
### ==============================================================================


In [ ]:
# -*- coding: utf-8 -*-
import xarray as xr
import numpy as np
import pandas as pd
import gsw
import matplotlib.pyplot as plt
from pathlib import Path
import re
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION BIOSWOT
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")

DIRS_IN = {
    "BioSWOT": BASE_ROOT / "Data" / "processed_step1_highres_qc",
}
DIRS_OUT = {
    "BioSWOT": BASE_ROOT / "Data" / "processed_step3_lag",
}
# Create output directories if they don't exist
for d in DIRS_OUT.values(): d.mkdir(parents=True, exist_ok=True)

MODELS_CSV = BASE_ROOT / "Figures" / "STEP2_LAG_ANALYSIS_FINAL" / "lag_models_step2.csv"

# Specific MVP IDs to plot for visual diagnostic checks
IDS_A_VISUALIZAR = ["0020", "0050", "0100"]

# ==========================================
# 2. FUNCTIONS
# ==========================================
def get_mvp_id(filename):
    """Extracts the 4-digit MVP profile ID from the filename."""
    match = re.search(r'mvp_(\d{4})', filename)
    return match.group(1) if match else None

def shift_signal_interp(data, lag_scans):
    """Shifts a 1D data array by a fractional number of scans using linear interpolation."""
    if np.isnan(lag_scans) or lag_scans == 0: return data
    x = np.arange(len(data))
    return np.interp(x, x + lag_scans, data, left=np.nan, right=np.nan)

def plot_diagnostic(p, t_raw, c_raw, s_raw, t_lag, s_lag, info):
    """Generates a 3-panel plot comparing raw vs. lag-corrected profiles."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 6), sharey=True, constrained_layout=True)
    
    # Panel A: Temperatura
    axs[0].plot(t_raw, p, 'k-', lw=1, alpha=0.3, label='Raw')
    axs[0].plot(t_lag, p, 'r--', lw=1.5, label='Lagged')
    axs[0].set_title("Temperature [°C]")
    axs[0].invert_yaxis()
    axs[0].legend()
    
    # Panel B: Conductividad
    axs[1].plot(c_raw, p, 'k-', lw=1, alpha=0.5)
    axs[1].set_title("Conductivity [mS/cm]")
    
    # Panel C: Salinidad
    axs[2].plot(s_raw, p, 'k-', lw=1, alpha=0.3, label='Raw Sal')
    axs[2].plot(s_lag, p, 'g-', lw=1.5, label='Lag-Corrected')
    axs[2].set_title("Practical Salinity")
    axs[2].legend()
    
    fig.suptitle(f"BioSWOT MVP {info['id']} | Regime: Standard | Lag Applied: {info['lag']:.2f} scans", 
                 fontsize=14, fontweight='bold', color='purple')
    plt.show()

# ==========================================
# 3. MAIN PROCESSING ROUTINE
# ==========================================
# Load the linear regression models generated in Step 2
models_df = pd.read_csv(MODELS_CSV)
models = {row['regime']: {'m': row['a'], 'c': row['b']} for _, row in models_df.iterrows()}

print(" Starting Step 3: Lag Application...")

for campaign, input_dir in DIRS_IN.items():
    files = sorted(list(input_dir.glob("*.nc")))
    print(f"Processing {len(files)} files in {campaign}...")
    
    for f in files:
        try:
            file_id = get_mvp_id(f.name)
            
            # BioSWOT uses only the model 'Standard'
            model = models.get('Standard')
            if model is None: continue
            
            with xr.open_dataset(f) as ds:
                if 't1' not in ds: continue
                
                p, t_raw, c_raw, w = ds.pressure.values, ds.t1.values, ds.c1.values, ds.w_descent.values
                
                # Calculate mean descent rate (w) for the profile's valid depth range
                mask_w = (p > 5) & np.isfinite(w)
                w_mean = np.mean(w[mask_w]) if mask_w.sum() > 20 else 0
                
                # Calculate the final lag in scans to apply lag = m * w + c
                lag_scans = model['m'] * w_mean + model['c']
                
                # Apply the shift to Temperature
                t_lagged = shift_signal_interp(t_raw, -lag_scans)
                
                # Calculate Salinity using the GSW library for both raw and lagged states
                s_raw = gsw.SP_from_C(c_raw, t_raw, p)
                s_lagged = gsw.SP_from_C(c_raw, t_lagged, p)
                
                # Plot diagnostic figure if the ID is in the watch list
                if file_id in IDS_A_VISUALIZAR:
                    info = {'id': file_id, 'lag': -lag_scans}
                    plot_diagnostic(p, t_raw, c_raw, s_raw, t_lagged, s_lagged, info)

                # Generación del archivo corregido
                ds_out = ds.copy(deep=True)
                ds_out['t1_lagged'] = (('scan',), t_lagged)
                ds_out['salinity_lagged'] = (('scan',), s_lagged)
                
                ds_out['salinity_lagged'].attrs = {
                    'units': 'PSU',
                    'lag_applied': float(-lag_scans),
                    'comment': 'Lag correction applied before Step 4'
                }
                
                out_name = f.name.replace("_step1_qc.nc", "_step3_lag.nc")
                ds_out.to_netcdf(DIRS_OUT[campaign] / out_name)
                ds.close()
                
        except Exception as e:
            print(f"❌ Error processing {f.name}: {e}")

print("Step 3 Completed. All profiles lag-corrected.")

🚀 Iniciando Paso 3 BioSWOT...
Procesando 633 archivos en BioSWOT...
✨ Hecho. Archivos generados en la carpeta processed_step3_lag.


In [ ]:
# Some diagnosis plots for visual checks

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import gsw
from pathlib import Path
import random
import warnings

warnings.filterwarnings("ignore")


BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")
STEP1_DIR = BASE_ROOT / "Data" / "processed_step1_highres_qc"
STEP3_DIR = BASE_ROOT / "Data" / "processed_step3_lag" 
OUTDIR = BASE_ROOT / "Figures" / "DIAG_VISUAL_CHECK_FINAL"
OUTDIR.mkdir(parents=True, exist_ok=True)

N_PLOTS = 8        
P_MAX_PLOT = 300   # maximum depth for plotting

# ==========================================
# 2. FUNCTIONS
# ==========================================
def get_arrays_step1(ds):
    if 'pressure' not in ds: return None, None, None, None
    p = ds['pressure'].values
    t = ds['t1'].values
    c = ds['c1'].values
    sp = gsw.SP_from_C(c, t, p)
    return p, t, c, sp

def get_arrays_step3(ds):
    if 'pressure' not in ds: return None, None, None, None
    p = ds['pressure'].values
    
    t_corr = ds['t1_lagged'].values if 't1_lagged' in ds else ds['t1'].values
    sp_corr = ds['salinity_lagged'].values if 'salinity_lagged' in ds else np.full_like(p, np.nan)
    
    c_raw = ds['c1'].values
    return p, t_corr, c_raw, sp_corr

def calculate_w_mean(ds):
    try:
        if 'w_descent' in ds:
            w = ds['w_descent'].values
            p = ds['pressure'].values
            mask = (p > 5) & np.isfinite(w)
            if mask.sum() > 10:
                return np.mean(w[mask])
    except: pass
    return 0.0

# ==========================================
# 3. FIGURE
# ==========================================
print(f" Searching for processed files in: {STEP3_DIR}")
files_s3 = sorted(list(STEP3_DIR.glob("*_step3_lag.nc"))) 

if not files_s3:
    print(f"❌ No files found. Verify that Step 3 saves files in: {STEP3_DIR}")
else:
    sample_size = min(len(files_s3), N_PLOTS)
    selected_files = random.sample(files_s3, sample_size)
    print(f"🎲 Generating figures for {len(selected_files)} random profiles...")
    
    for f3_path in selected_files:
        try:
            # Find the corresponding Raw file
            name_s1 = f3_path.name.replace("_step3_lag.nc", "_step1_qc.nc")
            f1_path = STEP1_DIR / name_s1
            if not f1_path.exists(): continue

            with xr.open_dataset(f1_path) as ds1, xr.open_dataset(f3_path) as ds3:
                p1, t1, c1, sp1 = get_arrays_step1(ds1)
                p3, t3, c3, sp3 = get_arrays_step3(ds3)
                
                if np.sum(np.isfinite(sp3)) < 10: continue
                
                # Retrieve information of the applied lag from the metadata
                lag_val = ds3['salinity_lagged'].attrs.get("lag_applied", np.nan)
                w_val = calculate_w_mean(ds3)
                
                # --- PLOT ---
                fig, axs = plt.subplots(1, 3, figsize=(16, 8), sharey=True, constrained_layout=True)
                
                # Temperature
                axs[0].plot(t1, p1, color='#AAAAAA', lw=2, alpha=0.6, label='Raw')
                axs[0].plot(t3, p3, color='tab:red', lw=1.5, ls='--', label='Lag-Corrected')
                axs[0].set_title("Temperature [°C]", fontweight='bold')
                axs[0].legend()

                # Conductivity
                axs[1].plot(c1, p1, color='#AAAAAA', lw=2, alpha=0.6)
                axs[1].set_title("Conductivity [mS/cm]", fontweight='bold')

                # Salinity
                axs[2].plot(sp1, p1, color='#AAAAAA', lw=2, alpha=0.6, label='Raw (Spiky)')
                axs[2].plot(sp3, p3, color='tab:green', lw=1.5, label='Corrected')
                axs[2].set_title("Practical Salinity [PSU]", fontweight='bold')
                axs[2].legend()
                
                for ax in axs:
                    ax.invert_yaxis()
                    ax.set_ylim(P_MAX_PLOT, 0)
                    ax.grid(True, alpha=0.3, ls=':')
                
                file_id = f3_path.stem.replace("_step3_lag", "")
                st = f"BioSWOT QC Check: {file_id}\nLag Applied: {lag_val:.3f} scans | Avg. Speed (w): {w_val:.2f} dbar/s"
                plt.suptitle(st, fontweight='bold', fontsize=14, color='purple')
                
                out_file = OUTDIR / f"Check_BioSWOT_{file_id}.png"
                plt.savefig(out_file, dpi=150)
                plt.close()
                print(f"  ✅ Plot saved: {file_id}")

        except Exception as e:
            print(f"❌ Error in profile {f3_path.name}: {e}")

print(f"\n ¡Finalized! Check the figures in: {OUTDIR}")

🔍 Buscando archivos procesados en: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\processed_step3_lag
🎲 Generando figuras para 8 perfiles aleatorios...
  ✅ Plot guardado: PL24_mvp_2023-05-14_002121
  ✅ Plot guardado: PL13_mvp_2023-05-01_035404
  ✅ Plot guardado: PL8_mvp_2023-04-29_104208
  ✅ Plot guardado: PL22_mvp_2023-05-12_084833
  ✅ Plot guardado: PL21_mvp_2023-05-10_030737
  ✅ Plot guardado: PL21_mvp_2023-05-10_050810
  ✅ Plot guardado: PL19_mvp_2023-05-09_094910
  ✅ Plot guardado: PL24_mvp_2023-05-14_042536

🎉 ¡Finalizado! Revisa las figuras en: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Figures\DIAG_VISUAL_CHECK_FINAL
